In [0]:
print("")   
%pip install nltk

In [1]:
### VECTORIZATION **TF-IDF** with library `scikit-learn`

### Código de Vectorización TF-IDF


from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import pandas as pd
from pathlib import Path
import string
import nltk
import re




# 1. Load the Dataset 
from pathlib import Path
DATA_DIR = Path("../data") #./data Actual level ---- ../data 2 levels up
# DATA_DIR = Path("/dbfs/workspace/datasets/imdb_big_dataset")
movies_path = DATA_DIR / "movies_final.csv"


# 0. Loading datasets imbd and links 
# Those two datasets are the conection between each movie and its information
movies_df = pd.read_csv(movies_path)
# movies_final = pd.read_csv("/dbfs/mnt/tu_ruta/data/movies_clean.csv")

# 1. Clean stopwords
stopwords = nltk.corpus.stopwords.words('english')
def clean_text(text):
    text = "".join([word for word in text if word not in string.punctuation])
    tokens = re.split('\\W+', text)
    text = [word for word in tokens if word not in stopwords]
    return text

#Apply the function
movies_df['combined_features_nostop'] = movies_df['combined_features'].apply(lambda x: clean_text(x.lower()))
movies_df.head(5)

,title,genres_list,overview,keywords,Cast_list,Director,vote_average,release_date,original_language,movieId,combined_features,combined_features_nostop
0,Inception,Action Science Fiction Adventure,"Cobb, a skilled thief who commits corporate es...",rescue mission dream airplane paris france vir...,Tim Kelleher Silvie Laguna Natasha Beaumont Kr...,Christopher Nolan,8.364,2010-07-15,en,79132,79132 Inception Action Science Fiction Adventu...,"[79132, inception, action, science, fiction, a..."
1,Interstellar,Adventure Drama Science Fiction,The adventures of a group of explorers who mak...,rescue future spacecraft race against time art...,Jeff Hephner William Devane Elyes Gabel Topher...,Christopher Nolan,8.417,2014-11-05,en,109487,109487 Interstellar Adventure Drama Science Fi...,"[109487, interstellar, adventure, drama, scien..."
2,The Dark Knight,Drama Action Crime Thriller,Batman raises the stakes in his war on crime. ...,joker sadism chaos secret identity crime fight...,Tommy Lister Jr. Edison Chen Beatrice Rosen To...,Christopher Nolan,8.512,2008-07-16,en,58559,58559 The Dark Knight Drama Action Crime Thril...,"[58559, dark, knight, drama, action, crime, th..."
3,Avatar,Action Adventure Fantasy Science Fiction,"In the 22nd century, a paraplegic Marine is di...",future society culture clash space travel spac...,Carvon Futrell Joel David Moore Jon Curry Laz ...,James Cameron,7.573,2009-12-15,en,72998,72998 Avatar Action Adventure Fantasy Science ...,"[72998, avatar, action, adventure, fantasy, sc..."
4,The Avengers,Science Fiction Action Adventure,When an unexpected enemy emerges and threatens...,new york city superhero shield based on comic ...,Haneyuri Nako Mizusawa Marin Rikako Sakata Ai ...,Joss Whedon,7.710,2012-04-25,en,89745,89745 The Avengers Science Fiction Action Adve...,"[89745, avengers, science, fiction, action, ad..."


In [2]:
print(len(movies_df.columns))
movies_df.columns

12


Index(['title', 'genres_list', 'overview', 'keywords', 'Cast_list', 'Director',
       'vote_average', 'release_date', 'original_language', 'movieId',
       'combined_features', 'combined_features_nostop'],
      dtype='str')

In [3]:
ps = nltk.PorterStemmer()

def stemming(tokenized_text):
    text = [ps.stem(word) for word in tokenized_text]
    return text

movies_df['combined_features_stemmed'] = movies_df['combined_features_nostop'].apply(lambda x: stemming(x))

# movies_df.head(5)

In [4]:
len(movies_df.iloc[0,len(movies_df.columns) - 1])
# print(movies_df.iloc[12])

print("combined_features") 
print(movies_df.iloc[0, 10])

print("\ncombined_features_nostop") #Punctuation
print(movies_df.iloc[0, 11])

print("\ncombined_features_stemmed") #Steemed
print(movies_df.iloc[0, 12])

# movies_df.iloc['combined_features_nostop']
# movies_df.iloc['combined_features']


combined_features
79132 Inception Action Science Fiction Adventure Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as payment for a task considered to be impossible: "inception", the implantation of another person's idea into a target's subconscious. rescue mission dream airplane paris france virtual reality kidnapping philosophy spy allegory manipulation car crash heist memory architecture los angeles california dream world subconscious Christopher Nolan Tim Kelleher Silvie Laguna Natasha Beaumont Kraig Thornber Jack Murray Adam Cole Claire Geare Marion Cotillard Magnus Nolan Tai-Li Lee Shannon Welles Taylor Geare Tom Berenger Coralie Dedykere Carl Gilliard Miranda Nolan Earl Cameron Yuji Okumoto Helena Cullinan Nicolas Clerc Andrew Pleavin Alex Lombard Mark Fleischmann Michael Gaston Marc Raducci Jack Gilroy Nicole Pulliam Shelley Lang Lukas Haas Russ Fega Felix Scott Ryan Hayward Cillian

In [5]:
# Revisar tipos en una columna específica
tipo_por_fila = movies_df["combined_features_stemmed"].map(type)
print(tipo_por_fila.value_counts())

combined_features_stemmed
<class 'list'>    86906
Name: count, dtype: int64


# LIMPIAR LISTAS EN STEEMED PARA PODER VECTORIZAR 

# 3. VECTORIZATION TF-IDF

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = movies_df["combined_features_stemmed"].fillna("").astype(str)
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

In [12]:
# # 2. Start Vectorizator TF-IDF with N-gram = 2 (e.g. "science fiction")
#Convierte cada lista de tokens a un string antes de vectorizar:


corpus = movies_df["combined_features_stemmed"] \
    .fillna("") \
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=0.01,
    max_df=0.8
)

tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)
print(f"Matriz generada. Forma: {tfidf_matrix.shape}")
type(tfidf_matrix)

Matriz generada. Forma: (86906, 1032)


scipy.sparse._csr.csr_matrix

In [ ]:

# 3. Ajustar y transformar los datos
# # Usamos la columna 'combined_features_stemmed' que generamos en el paso anterior
# Usar la variable `corpus` (listas convertidas a strings) creada en la celda anterior
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

print(f"Matriz generada con éxito. Forma de la matriz: {tfidf_matrix.shape}")

# # 4. Guardar los artefactos en Databricks para su uso posterior en el chatbot
# # Es crucial guardar tanto la matriz como el vectorizador para procesar las consultas del usuario [3, 4]
# joblib.dump(tfidf_vectorizer, '/dbfs/mnt/tu_ruta/models/tfidf_vectorizer.pkl')
# joblib.dump(tfidf_matrix, '/dbfs/mnt/tu_ruta/models/tfidf_matrix.pkl')

# print("Artefactos guardados en la carpeta /models/")

AttributeError: 'list' object has no attribute 'lower'